# Biomechatronic Prosthetic Hand: NinaPro DB8 Causal TCN Pipeline
This notebook trains and evaluates a causal dilated Temporal Convolutional Network (TCN) mapping 16-channel sEMG (2 kHz) to continuous finger kinematics using NinaPro Database 8 (DB8).

### Key Scientific Protocols
- **Strict Held-Out Split:** Sessions 1 & 2 (A1 & A2) for training and validation; Session 3 (A3) is strictly held out.
- **No Future Leakage:** Causal IIR filtering and causal left-padded 1D dilated convolutions.
- **Normalization Isolation:** sEMG Z-score and CyberGlove scaling fitted strictly on Training Acquisition A1.

In [ ]:
# 1. Install dependencies (for Colab environments)
!pip install -q pyyaml scipy pandas matplotlib onnx
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Download / Mount NinaPro DB8 Subject 1 Data
import os, urllib.request
os.makedirs("data/ninapro_db8", exist_ok=True)
base_url = "https://ninapro.hevs.ch/files/DB8/"
files = ["S1_E1_A1.mat", "S1_E1_A2.mat", "S1_E1_A3.mat"]
for f in files:
    dest = os.path.join("data/ninapro_db8", f)
    if not os.path.exists(dest):
        print(f"Downloading {f}...")
        urllib.request.urlretrieve(base_url + f, dest)
        print(f"Saved {f} ({os.path.getsize(dest)/(1024*1024):.1f} MB)")
    else:
        print(f"{f} already present ({os.path.getsize(dest)/(1024*1024):.1f} MB)")

In [ ]:
# 3. Dataset Audit & Structure Inspection
import scipy.io as sio
for a in ["A1", "A2", "A3"]:
    mat = sio.loadmat(f"data/ninapro_db8/S1_E1_{a}.mat")
    print(f"{a}: emg={mat['emg'].shape}, glove={mat['glove'].shape}, reps={len(set(mat['repetition'].flatten()))}")

In [ ]:
# 4. Clone or Import Pipeline Modules
# If running standalone in Colab, ensure modules are in PYTHONPATH
import sys
sys.path.append(".")
from ml.dataset import NinaProDB8SessionDataset
from ml.windowing import WindowConfig
from ml.model import CausalTCN
from ml.train import main as run_train
from ml.benchmark import run_full_benchmark
from ml.quantize import evaluate_quantization

In [ ]:
# 5. Train Model & Evaluate on Held-Out Session 3
run_train("configs/tcn_db8.yaml")

In [ ]:
# 6. Benchmark Latency and Model Size
bench_results = run_full_benchmark("results/best_model.pt")

In [ ]:
# 7. Model Quantization and ONNX Export for TinyML Deployment
quant_results = evaluate_quantization("results/best_model.pt")

In [ ]:
# 8. Display Final Session 3 Results
import json
with open("results/session3_test_results.json", "r") as f:
    res = json.load(f)
print(json.dumps(res, indent=2))